# 02 - Frame Sampling with Background Subtraction
Extract candidate frames containing animals using background subtraction.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

DRIVE_RAW_VIDEOS = "/content/drive/My Drive/PigBehavior/raw_videos"
DRIVE_LABELED = "/content/drive/My Drive/PigBehavior/labeled_frames"
DRIVE_BACKGROUNDS = "/content/drive/My Drive/PigBehavior/backgrounds"

# Sampling settings
FRAMES_PER_VIDEO = 30

# Background subtraction parameters
BG_THRESHOLD = 30       # pixel intensity difference to count as foreground
MIN_ANIMAL_PIXELS = 500  # minimum foreground pixels to consider animal present
BG_SAMPLE_EVERY_N = 100  # sample every Nth frame for background computation
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!apt-get install -y -qq tesseract-ocr > /dev/null 2>&1
!pip install --quiet opencv-python pandas numpy matplotlib pytesseract

In [ ]:
from pathlib import Path
from src.io.video_inventory import scan_videos
from src.pose.frame_sampler import (
    compute_background, detect_animal,
    sample_with_background_subtraction, save_frame
)

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"\nFound {len(df)} usable videos")
df[["filename", "session", "camera", "duration_min"]]

In [ ]:
import cv2
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

bg_dir = Path(DRIVE_BACKGROUNDS)
bg_dir.mkdir(parents=True, exist_ok=True)

# Compute one background per video
backgrounds = {}
for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing backgrounds"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    key = f"{row['session']}_cam{row['camera']}"
    bg_path = bg_dir / f"{key}_bg.jpg"
    if bg_path.exists():
        backgrounds[key] = cv2.imread(str(bg_path))
        continue
    bg = compute_background(str(video_path), BG_SAMPLE_EVERY_N)
    if bg is not None:
        cv2.imwrite(str(bg_path), bg)
        backgrounds[key] = bg

print(f"Computed {len(backgrounds)} backgrounds")

In [ ]:
# Preview a few backgrounds
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (key, bg) in zip(axes.flat, list(backgrounds.items())[:8]):
    ax.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB))
    ax.set_title(key)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np

total_saved = 0
total_skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Sampling frames"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    camera = row["camera"]
    session = row["session"]
    stem = Path(row["filename"]).stem
    key = f"{session}_cam{camera}"

    bg = backgrounds.get(key)
    if bg is None:
        print(f"No background for {key}, skipping")
        continue

    frames = sample_with_background_subtraction(
        str(video_path),
        FRAMES_PER_VIDEO,
        background=bg,
        threshold=BG_THRESHOLD,
        min_pixels=MIN_ANIMAL_PIXELS,
    )

    if not frames:
        print(f"No animal frames in {row['filename']}, falling back to uniform")
        from src.pose.frame_sampler import sample_uniform
        frames = sample_uniform(str(video_path), FRAMES_PER_VIDEO)
        total_skipped += 1

    for frame_idx, frame in frames:
        fname = save_frame(DRIVE_LABELED, session, camera, stem, frame_idx, frame)
        total_saved += 1

print(f"\nTotal frames saved: {total_saved} to {DRIVE_LABELED}")
if total_skipped:
    print(f"{total_skipped} videos had no detections (fallback used)")

In [ ]:
KEYPOINT_NAMES = ["snout", "left_ear", "right_ear", "neck",
                  "shoulders", "mid_back", "hip", "tail_base"]

print("=" * 60)
print("NEXT STEPS:")
print("1. Download the labeled_frames/ folder from Google Drive")
print("2. Label keypoints using LabelMe:")
print("   $ pip install labelme && labelme /path/to/labeled_frames")
print(f"3. For each image, create POINT annotations with these {len(KEYPOINT_NAMES)} labels:")
for i, kp in enumerate(KEYPOINT_NAMES):
    print(f"      {i+1}. {kp}")
print("4. Upload labeled_frames/ back to Drive")
print("5. Proceed to notebook 03_Pose_Training.ipynb")